In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install gensim wandb --quiet

In [3]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import wandb
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Using device: cuda


In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
 
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum())
print("\nAnswer label distribution:\n", train_df["answer"].value_counts())
 
train_df["prompt_len"] = train_df["prompt"].astype(str).apply(lambda x: len(x.split()))
print("\nPrompt word-length stats:\n", train_df["prompt_len"].describe())

print(train_df.head(3))

Train shape: (2000, 8)
Test shape : (500, 7)

Missing values in train:
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer label distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt word-length stats:
 count    2000.00000
mean       18.14650
std         6.78189
min         3.00000
25%        14.00000
50%        17.00000
75%        22.00000
max        51.00000
Name: prompt_len, dtype: float64
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   

                         

In [14]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
TEXT_COLS = ["prompt", "A", "B", "C", "D", "E"]
 
for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

In [15]:
def tokenize(text):
    return text.split()

In [16]:
all_sentences = []
for df in [train_df, test_df]:
    for col in TEXT_COLS:
        all_sentences.extend(df[col].apply(tokenize).tolist())
 
EMBED_DIM = 100
 
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=SEED,
)
 
print("Vocabulary size:", len(w2v_model.wv))
w2v_model.save(os.path.join(OUTPUT_DIR, "word2vec.model"))

Vocabulary size: 2973


In [17]:
def text_to_vector(text, model, dim=EMBED_DIM):
    words = tokenize(text)
    vecs = []

    for word in words:
        if word in model.wv:
            vecs.append(model.wv[word])

    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)

    avg_vector = np.mean(vecs, axis=0)
    return avg_vector.astype(np.float32)

In [18]:
LABELS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

class MCQDataset(Dataset):

    def __init__(self, df, w2v_model, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.model = w2v_model
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        question_vector = text_to_vector(row["prompt"], self.model)

        features = []

        for option in LABELS:
            option_vector = text_to_vector(row[option], self.model)

            difference = np.abs(question_vector - option_vector)

            feature = np.concatenate((question_vector, option_vector, difference))

            features.append(feature)

        features = np.array(features)

        data = {}
        data["features"] = torch.tensor(features, dtype=torch.float32)

        if self.has_labels:
            answer = LABEL2IDX[row["answer"]]
            data["label"] = torch.tensor(answer, dtype=torch.long)
        else:
            data["id"] = row["id"]

        return data

In [21]:
class MCQScorer(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
 
    def forward(self, x):
        batch, n_options, dim = x.shape
        x = x.view(batch * n_options, dim)
        scores = self.net(x)
        scores = scores.view(batch, n_options)
        return scores

In [22]:
def map_at_3(probs, labels):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = []

    for pred, true in zip(top3, labels):
        if true in pred:
            score.append(1 / (np.where(pred == true)[0][0] + 1))
        else:
            score.append(0)

    return np.mean(score)


def run_epoch(model, loader, optimizer, criterion, train=True):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    for batch in loader:

        x = batch["features"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(train):

            output = model(x)
            loss = criterion(output, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)

        all_probs.append(torch.softmax(output, dim=1).cpu().detach().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    loss = total_loss / len(loader.dataset)
    acc = (all_probs.argmax(1) == all_labels).mean()
    map3 = map_at_3(all_probs, all_labels)

    return loss, acc, map3


try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except:
    pass

wandb.login()

wandb.init(
    project="dlgenai-project-26t2",
    config={
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "hidden_dim": 128
    }
)

cfg = wandb.config

train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["answer"]
)

train_loader = DataLoader(
    MCQDataset(train_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    MCQDataset(val_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=False
)

model = MCQScorer(3 * EMBED_DIM, cfg.hidden_dim).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

best_map = 0

for epoch in range(cfg.epochs):

    train_loss, train_acc, train_map = run_epoch(
        model, train_loader, optimizer, criterion, True
    )

    val_loss, val_acc, val_map = run_epoch(
        model, val_loader, optimizer, criterion, False
    )

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_map3": train_map,
        "val_map3": val_map
    })

    print(f"Epoch {epoch+1}  Validation MAP@3 = {val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), OUTPUT_DIR + "/best_model.pt")

wandb.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Epoch 1  Validation MAP@3 = 0.6500
Epoch 2  Validation MAP@3 = 0.6906
Epoch 3  Validation MAP@3 = 0.7583
Epoch 4  Validation MAP@3 = 0.7867
Epoch 5  Validation MAP@3 = 0.8028
Epoch 6  Validation MAP@3 = 0.8439
Epoch 7  Validation MAP@3 = 0.8622
Epoch 8  Validation MAP@3 = 0.8678
Epoch 9  Validation MAP@3 = 0.8833
Epoch 10  Validation MAP@3 = 0.8894
Epoch 11  Validation MAP@3 = 0.8867
Epoch 12  Validation MAP@3 = 0.9011
Epoch 13  Validation MAP@3 = 0.9289
Epoch 14  Validation MAP@3 = 0.9239
Epoch 15  Validation MAP@3 = 0.9339
Epoch 16  Validation MAP@3 = 0.9306
Epoch 17  Validation MAP@3 = 0.9222
Epoch 18  Validation MAP@3 = 0.9356
Epoch 19  Validation MAP@3 = 0.9272
Epoch 20  Validation MAP@3 = 0.9372


train_loss,█▇▅▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train_map3,▁▃▄▅▅▆▆▇▇▇▇▇█▇██████
val_loss,█▆▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁
val_map3,▁▂▄▄▅▆▆▆▇▇▇▇████████
train_loss,0.54427
train_map3,0.86324
val_loss,0.3929
val_map3,0.93722


In [23]:
model = MCQScorer(3 * EMBED_DIM, 128).to(DEVICE)
model.load_state_dict(torch.load(OUTPUT_DIR + "/best_model.pt", map_location=DEVICE))
model.eval()

test_loader = DataLoader(
    MCQDataset(test_df, w2v_model, has_labels=False),
    batch_size=32,
    shuffle=False
)

ids = []
predictions = []

with torch.no_grad():

    for batch in test_loader:

        x = batch["features"].to(DEVICE)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()
        top3 = np.argsort(-probs, axis=1)[:, :3]

        for i in range(len(top3)):
            ids.append(int(batch["id"][i]))
            predictions.append(" ".join(LABELS[j] for j in top3[i]))

submission = pd.DataFrame({
    "id": ids,
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("submission.csv created")

          id Prediction
0  tensor(1)      A C D
1  tensor(2)      B D C
2  tensor(3)      B E D
3  tensor(4)      E A C
4  tensor(5)      D C B
submission.csv created
